# TextCNN情感分类实验

使用去年实验的英文影评和MindSpore TextCNN。完成四处代码，再比较两组卷积核。操作步骤见实验手册。

## 1 检查环境

先在JupyterLab打开本目录下的data.py、model.py、train.py，填写TODO 1至4。my_reviews.txt另写两句英文影评。下面的命令均调用当前Kernel的Python，修改py文件后重新运行对应单元格即可。

In [ ]:
import sys
import subprocess
from pathlib import Path
import mindspore as ms
print("Python", sys.version)
print("MindSpore", ms.__version__)
print("当前目录", Path.cwd())
DEVICE = "GPU"  # 与实例硬件一致，CPU或Ascend实例改为对应名称
assert Path("train.py").is_file(), "请在textcnn_lab目录打开Notebook"

## 2 检查数据

随包正式数据共10662条。prepare_data.py会核对每类5331条，文件齐全时直接读取。

In [ ]:
subprocess.run([sys.executable, "prepare_data.py"], check=True)
subprocess.run([sys.executable, "data.py"], check=True)

## 3 分别检查四处代码

每完成一处就执行对应命令。也可以将下面的--task参数依次改为1、2、3、4。

In [ ]:
subprocess.run([sys.executable, "check_tasks.py", "--task", "1"], check=True)
subprocess.run([sys.executable, "check_tasks.py", "--device", DEVICE], check=True)

## 4 训练基线

卷积核3、4、5，各96通道，训练8轮。每轮输出训练损失、验证准确率和F1。全部结果写入results/baseline。

In [ ]:
subprocess.run([sys.executable, "train.py", "--device", DEVICE, "--run", "baseline"], check=True)

## 5 训练对照组

仅将卷积核换成2、3、4。词表、训练验证划分、种子和其他参数与基线相同。结果写入results/compare。

In [ ]:
subprocess.run([sys.executable, "train.py", "--device", DEVICE, "--run", "compare"], check=True)

## 6 查看结果并填写报告

两组均记录最后一轮指标。P、R、F1以正面评论1为正类。predictions.csv有两句给定影评和两句自拟影评。validation_predictions.csv可用于查看误判。修改自拟影评后可用--predict-only重新生成预测。

In [ ]:
import json
import csv
for name in ["baseline", "compare"]:
    folder = Path("results") / name
    metrics = json.loads((folder / "metrics.json").read_text(encoding="utf-8"))
    print(name, {key: metrics[key] for key in ["accuracy", "precision", "recall", "f1"]})
    with (folder / "predictions.csv").open(encoding="utf-8-sig") as stream:
        print(list(csv.DictReader(stream)))

## 7 提交

将Word实验报告填写完成，保存在本目录。运行下面的单元格后，下载submission.zip，改名为“学号_姓名_TextCNN.zip”。提交包包含代码、Notebook、两组结果和报告。模型权重保留在云端工程中。

In [ ]:
subprocess.run([sys.executable, "make_submission.py"], check=True)